# GemiCenterSingle (Restart-Safe, Checkpointed)

This notebook is a clear, minimal pipeline for single-source frequency transfer.

It is restart-safe by design:
- Hyperparameter results (`hp_df`, `best_hp`) are saved to disk.
- Directional models are checkpointed per `(direction, n_train, width, lr, wd, cfg signature)`.
- Re-running after a kernel restart loads cached artifacts when available.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import product
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse.linalg as spla

from itertools import product
from tqdm.auto import tqdm
import time


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120


In [2]:
@dataclass
class RunCfg:
    seed: int = 42
    n_tot: int = 500
    npml: int = 104
    pml_eta: float = 70.0
    pml_power: float = 2.0
    stencil_order: int = 2
    omega_low: float = 32.0
    omega_high: float = 64.0
    source_margin: int = 16
    amp_min: float = 1.0
    amp_max: float = 1.0
    n_train: int = 240
    n_val: int = 60
    n_test: int = 60
    batch_size: int = 8
    epochs: int = 120
    lr: float = 1e-3
    weight_decay: float = 1e-6
    scheduler_patience: int = 10
    scheduler_factor: float = 0.5
    early_stop_patience: int = 20
    grad_clip: float = 1.0

cfg = RunCfg()
rng = np.random.default_rng(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ART = Path("artifacts/gemi_center_single")
ART.mkdir(parents=True, exist_ok=True)
BEST_HP_PATH = ART / "best_hp.json"
HP_DF_PATH = ART / "hp_df.csv"

print("device:", device)
print("artifacts:", ART.resolve())
print(cfg)


device: cpu
artifacts: /math/home/fkiewiet/Freq2Transfer/experiments/artifacts/gemi_center_single
RunCfg(seed=42, n_tot=500, npml=104, pml_eta=70.0, pml_power=2.0, stencil_order=2, omega_low=32.0, omega_high=64.0, source_margin=16, amp_min=1.0, amp_max=1.0, n_train=240, n_val=60, n_test=60, batch_size=8, epochs=120, lr=0.001, weight_decay=1e-06, scheduler_patience=10, scheduler_factor=0.5, early_stop_patience=20, grad_clip=1.0)


/math/home/fkiewiet/Freq2Transfer/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1) Helmholtz operators and helpers

In [3]:
def sigma_profile_1d(n_tot: int, n_pml: int, eta: float, pml_power: float) -> np.ndarray:
    sig = np.zeros(n_tot, dtype=float)
    if n_pml <= 0:
        return sig
    for i in range(n_tot):
        if i < n_pml:
            dist = (n_pml - i) / n_pml
        elif i >= n_tot - n_pml:
            dist = (i - (n_tot - n_pml) + 1) / n_pml
        else:
            dist = 0.0
        sig[i] = eta * (dist ** pml_power)
    return sig


def get_helmholtz_matrix(*, omega: float, n_tot: int, n_pml: int, eta: float, pml_power: float, stencil_order: int = 2):
    if stencil_order not in (2, 4):
        raise ValueError("stencil_order must be 2 or 4")
    h = 1.0 / (n_tot - 1)
    sig = sigma_profile_1d(n_tot, n_pml, eta, pml_power)
    s = 1.0 / (1.0 + 1j * sig / (omega / (2.0 * np.pi)))

    rows, cols, vals = [], [], []

    def add(r, c, v):
        rows.append(r)
        cols.append(c)
        vals.append(v)

    def lin(i, j):
        return i * n_tot + j

    d2, o21 = -2.0, 1.0
    d4, o41, o42 = -2.5, 4.0 / 3.0, -1.0 / 12.0
    k2 = float(omega) * float(omega)

    for i in range(n_tot):
        sy2 = s[i] * s[i]
        for j in range(n_tot):
            r = lin(i, j)
            if i == 0 or j == 0 or i == n_tot - 1 or j == n_tot - 1:
                add(r, r, 1.0 + 0.0j)
                continue

            sx2 = s[j] * s[j]
            use_2nd = (stencil_order == 2 or i < 2 or j < 2 or i > n_tot - 3 or j > n_tot - 3)
            if use_2nd:
                add(r, r, ((sx2 * d2 + sy2 * d2) / (h * h)) + k2)
                add(r, lin(i, j - 1), sx2 * o21 / (h * h))
                add(r, lin(i, j + 1), sx2 * o21 / (h * h))
                add(r, lin(i - 1, j), sy2 * o21 / (h * h))
                add(r, lin(i + 1, j), sy2 * o21 / (h * h))
            else:
                add(r, r, ((sx2 * d4 + sy2 * d4) / (h * h)) + k2)
                add(r, lin(i, j - 1), sx2 * o41 / (h * h)); add(r, lin(i, j + 1), sx2 * o41 / (h * h))
                add(r, lin(i, j - 2), sx2 * o42 / (h * h)); add(r, lin(i, j + 2), sx2 * o42 / (h * h))
                add(r, lin(i - 1, j), sy2 * o41 / (h * h)); add(r, lin(i + 1, j), sy2 * o41 / (h * h))
                add(r, lin(i - 2, j), sy2 * o42 / (h * h)); add(r, lin(i + 2, j), sy2 * o42 / (h * h))

    return sp.coo_matrix((vals, (rows, cols)), shape=(n_tot * n_tot, n_tot * n_tot)).tocsr()


def make_single_source_rhs(n_tot: int, n_pml: int, margin: int, amp_min: float, amp_max: float, rng_local: np.random.Generator):
    f = np.zeros((n_tot, n_tot), dtype=np.complex128)
    lo = n_pml + margin
    hi = n_tot - n_pml - margin
    if hi <= lo:
        raise ValueError("source margin too large")
    i = int(rng_local.integers(lo, hi))
    j = int(rng_local.integers(lo, hi))
    amp = float(rng_local.uniform(amp_min, amp_max))
    phase = float(rng_local.uniform(0.0, 2 * np.pi))
    f[i, j] = amp * np.exp(1j * phase)
    return f


def solve_field(A: sp.csr_matrix, f2d: np.ndarray):
    u = spla.spsolve(A, f2d.reshape(-1))
    return u.reshape(f2d.shape)


def to_2ch(arr: np.ndarray) -> np.ndarray:
    return np.stack([arr.real, arr.imag], axis=0).astype(np.float32)


def from_2ch(arr2: np.ndarray) -> np.ndarray:
    return arr2[0] + 1j * arr2[1]


In [4]:
A32 = get_helmholtz_matrix(
    omega=cfg.omega_low,
    n_tot=cfg.n_tot,
    n_pml=cfg.npml,
    eta=cfg.pml_eta,
    pml_power=cfg.pml_power,
    stencil_order=cfg.stencil_order,
)
A64 = get_helmholtz_matrix(
    omega=cfg.omega_high,
    n_tot=cfg.n_tot,
    n_pml=cfg.npml,
    eta=cfg.pml_eta,
    pml_power=cfg.pml_power,
    stencil_order=cfg.stencil_order,
)

f0 = make_single_source_rhs(cfg.n_tot, cfg.npml, cfg.source_margin, cfg.amp_min, cfg.amp_max, rng)
u32_0 = solve_field(A32, f0)
u64_0 = solve_field(A64, f0)
r32 = f0.reshape(-1) - A32 @ u32_0.reshape(-1)
r64 = f0.reshape(-1) - A64 @ u64_0.reshape(-1)
print("||r32||2/||f||2 =", np.linalg.norm(r32) / max(np.linalg.norm(f0), 1e-30))
print("||r64||2/||f||2 =", np.linalg.norm(r64) / max(np.linalg.norm(f0), 1e-30))


||r32||2/||f||2 = 2.420640689484591e-14
||r64||2/||f||2 = 7.95191377483663e-14


## 2) Dataset and preprocessing

In [5]:
def build_transfer_dataset(n_samples: int, seed: int, cfg_local: RunCfg):
    rng_local = np.random.default_rng(seed)
    X_up, Y_up = [], []
    X_dn, Y_dn = [], []
    for _ in range(n_samples):
        f = make_single_source_rhs(
            cfg_local.n_tot,
            cfg_local.npml,
            cfg_local.source_margin,
            cfg_local.amp_min,
            cfg_local.amp_max,
            rng_local,
        )
        u32 = solve_field(A32, f)
        u64 = solve_field(A64, f)
        X_up.append(to_2ch(u32)); Y_up.append(to_2ch(u64))
        X_dn.append(to_2ch(u64)); Y_dn.append(to_2ch(u32))

    return {
        "X_up": np.asarray(X_up, dtype=np.float32),
        "Y_up": np.asarray(Y_up, dtype=np.float32),
        "X_down": np.asarray(X_dn, dtype=np.float32),
        "Y_down": np.asarray(Y_dn, dtype=np.float32),
    }


def window_nonpml(X: np.ndarray, npml: int):
    if npml <= 0:
        return X
    return X[:, :, npml:-npml, npml:-npml]


def channel_scale_fit(X: np.ndarray, eps: float = 1e-8):
    s = X.std(axis=(0, 2, 3)).astype(np.float32)
    return np.maximum(s, eps)


def channel_scale_apply(X: np.ndarray, s: np.ndarray):
    return X / s[None, :, None, None]


def channel_scale_undo(X: np.ndarray, s: np.ndarray):
    return X * s[None, :, None, None]


## 3) Model, train loop, metrics

In [6]:
class PlainCNN(nn.Module):
    def __init__(self, width: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, width, 3, padding=1), nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1), nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1), nn.GELU(),
            nn.Conv2d(width, 2, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


def train_one_direction(Xtr, Ytr, Xva, Yva, *, width: int, lr: float, weight_decay: float, cfg_local: RunCfg):
    model = PlainCNN(width=width).to(device)
    ds = TensorDataset(torch.tensor(Xtr), torch.tensor(Ytr))
    dl = DataLoader(ds, batch_size=cfg_local.batch_size, shuffle=True, drop_last=False)

    xv = torch.tensor(Xva, dtype=torch.float32, device=device)
    yv = torch.tensor(Yva, dtype=torch.float32, device=device)

    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=cfg_local.scheduler_factor, patience=cfg_local.scheduler_patience
    )

    best = {"epoch": -1, "val": float("inf"), "state": None}
    history = []
    no_improve = 0

    for ep in range(1, cfg_local.epochs + 1):
        model.train()
        tr_losses = []
        for xb, yb in dl:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = torch.mean((pred - yb) ** 2)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg_local.grad_clip)
            opt.step()
            tr_losses.append(float(loss.item()))

        model.eval()
        with torch.no_grad():
            pv = model(xv)
            val = float(torch.mean((pv - yv) ** 2).item())

        sched.step(val)
        tr = float(np.mean(tr_losses))
        history.append({"epoch": ep, "train_mse": tr, "val_mse": val, "lr": float(opt.param_groups[0]["lr"])})

        if val < best["val"]:
            best["val"] = val
            best["epoch"] = ep
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= cfg_local.early_stop_patience:
            break

    model.load_state_dict(best["state"])
    return model, history, best


def predict_np(model: nn.Module, X: np.ndarray, batch_size: int):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i + batch_size], dtype=torch.float32, device=device)
            yb = model(xb).detach().cpu().numpy()
            outs.append(yb)
    return np.concatenate(outs, axis=0)


def metrics_2ch(pred: np.ndarray, true: np.ndarray):
    e = pred - true
    return {
        "mse_re": float(np.mean((e[:, 0]) ** 2)),
        "mse_im": float(np.mean((e[:, 1]) ** 2)),
        "rel_l2_re": float(np.linalg.norm(e[:, 0].ravel()) / max(np.linalg.norm(true[:, 0].ravel()), 1e-30)),
        "rel_l2_im": float(np.linalg.norm(e[:, 1].ravel()) / max(np.linalg.norm(true[:, 1].ravel()), 1e-30)),
    }


## 4) Hyperparameter sweep (cached to disk)

In [ ]:
from itertools import product
from tqdm.auto import tqdm
import time
import json
import pandas as pd

def quick_val_score(
    direction: str,
    width: int,
    lr: float,
    wd: float,
    cfg_local: RunCfg,
    train_ds: dict,
    val_ds: dict,
) -> float:
    if direction == "up":
        Xtr, Ytr = train_ds["X_up"], train_ds["Y_up"]
        Xva, Yva = val_ds["X_up"], val_ds["Y_up"]
    elif direction == "down":
        Xtr, Ytr = train_ds["X_down"], train_ds["Y_down"]
        Xva, Yva = val_ds["X_down"], val_ds["Y_down"]
    else:
        raise ValueError(f"Unknown direction: {direction}")

    Xtr = window_nonpml(Xtr, cfg_local.npml)
    Ytr = window_nonpml(Ytr, cfg_local.npml)
    Xva = window_nonpml(Xva, cfg_local.npml)
    Yva = window_nonpml(Yva, cfg_local.npml)

    sx = channel_scale_fit(Xtr)
    sy = channel_scale_fit(Ytr)

    Xtr_n = channel_scale_apply(Xtr, sx)
    Ytr_n = channel_scale_apply(Ytr, sy)
    Xva_n = channel_scale_apply(Xva, sx)
    Yva_n = channel_scale_apply(Yva, sy)

    _, _, best = train_one_direction(
        Xtr_n, Ytr_n, Xva_n, Yva_n,
        width=width, lr=lr, weight_decay=wd, cfg_local=cfg_local
    )
    return float(best["val"])


if BEST_HP_PATH.exists() and HP_DF_PATH.exists():
    hp_df = pd.read_csv(HP_DF_PATH)
    best_hp = json.loads(BEST_HP_PATH.read_text())
    print("Loaded cached hp_df and best_hp")
else:
    train_ds = build_transfer_dataset(cfg.n_train, cfg.seed + 1, cfg)
    val_ds = build_transfer_dataset(cfg.n_val, cfg.seed + 2, cfg)

    grid = {
        "width": [24, 32, 48],
        "lr": [5e-4, 1e-3, 2e-3],
        "wd": [1e-6, 1e-5],
    }

    grid_list = list(product(grid["width"], grid["lr"], grid["wd"]))
    rows = []
    t0 = time.time()

    for i, (width, lr, wd) in enumerate(tqdm(grid_list, desc="HP sweep"), start=1):
        v_up = quick_val_score("up", width, lr, wd, cfg, train_ds, val_ds)
        v_dn = quick_val_score("down", width, lr, wd, cfg, train_ds, val_ds)
        v_mean = 0.5 * (v_up + v_dn)

        rows.append({
            "width": width,
            "lr": lr,
            "weight_decay": wd,
            "val_up": v_up,
            "val_down": v_dn,
            "val_mean": v_mean,
        })

        tqdm.write(
            f"[{i}/{len(grid_list)}] width={width}, lr={lr}, wd={wd}, "
            f"val_mean={v_mean:.4e}, elapsed={(time.time()-t0)/60:.1f}m"
        )

    hp_df = pd.DataFrame(rows).sort_values("val_mean").reset_index(drop=True)
    best_hp = hp_df.iloc[0].to_dict()

    hp_df.to_csv(HP_DF_PATH, index=False)
    BEST_HP_PATH.write_text(json.dumps(best_hp, indent=2))
    print("Saved hp_df and best_hp")

display(hp_df.head(10))
print("best_hp:", best_hp)


## 5) Dataset-size sweep with model checkpoint reuse

In [ ]:
def ckpt_name(direction: str, n_train: int, width: int, lr: float, wd: float, cfg_local: RunCfg) -> Path:
    key = {
        "direction": direction,
        "n_train": int(n_train),
        "width": int(width),
        "lr": float(lr),
        "wd": float(wd),
        "seed": int(cfg_local.seed),
        "n_tot": int(cfg_local.n_tot),
        "npml": int(cfg_local.npml),
        "omega_low": float(cfg_local.omega_low),
        "omega_high": float(cfg_local.omega_high),
        "stencil_order": int(cfg_local.stencil_order),
    }
    digest = hashlib.md5(json.dumps(key, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    return ART / f"model_{direction}_n{n_train}_{digest}.pt"


def get_or_train_model(direction: str, tr: dict, va: dict, cfg_local: RunCfg, width: int, lr: float, wd: float):
    if direction == "up":
        Xtr, Ytr = tr["X_up"], tr["Y_up"]
        Xva, Yva = va["X_up"], va["Y_up"]
    else:
        Xtr, Ytr = tr["X_down"], tr["Y_down"]
        Xva, Yva = va["X_down"], va["Y_down"]

    Xtr = window_nonpml(Xtr, cfg_local.npml)
    Ytr = window_nonpml(Ytr, cfg_local.npml)
    Xva = window_nonpml(Xva, cfg_local.npml)
    Yva = window_nonpml(Yva, cfg_local.npml)

    ckpt = ckpt_name(direction, cfg_local.n_train, width, lr, wd, cfg_local)

    if ckpt.exists():
        blob = torch.load(ckpt, map_location=device)
        model = PlainCNN(width=width).to(device)
        model.load_state_dict(blob["state_dict"])
        sx = np.asarray(blob["sx"], dtype=np.float32)
        sy = np.asarray(blob["sy"], dtype=np.float32)
        meta = {"checkpoint": str(ckpt), "loaded": True}
        return model, sx, sy, meta

    sx = channel_scale_fit(Xtr)
    sy = channel_scale_fit(Ytr)
    Xtr_n = channel_scale_apply(Xtr, sx)
    Ytr_n = channel_scale_apply(Ytr, sy)
    Xva_n = channel_scale_apply(Xva, sx)
    Yva_n = channel_scale_apply(Yva, sy)

    model, _, best = train_one_direction(
        Xtr_n, Ytr_n, Xva_n, Yva_n, width=width, lr=lr, weight_decay=wd, cfg_local=cfg_local
    )

    torch.save({
        "state_dict": model.state_dict(),
        "sx": sx,
        "sy": sy,
        "best_epoch": best["epoch"],
        "best_val": best["val"],
        "width": int(width),
        "lr": float(lr),
        "weight_decay": float(wd),
    }, ckpt)

    meta = {"checkpoint": str(ckpt), "loaded": False}
    return model, sx, sy, meta


def run_with_train_size(n_train: int, cfg_base: RunCfg, width: int, lr: float, wd: float):
    cfg_local = RunCfg(**{**cfg_base.__dict__, "n_train": int(n_train)})
    tr = build_transfer_dataset(cfg_local.n_train, cfg_local.seed + 11, cfg_local)
    va = build_transfer_dataset(cfg_local.n_val, cfg_local.seed + 12, cfg_local)
    te = build_transfer_dataset(cfg_local.n_test, cfg_local.seed + 13, cfg_local)

    def one(direction: str):
        if direction == "up":
            Xte, Yte = te["X_up"], te["Y_up"]
        else:
            Xte, Yte = te["X_down"], te["Y_down"]

        model, sx, sy, meta = get_or_train_model(direction, tr, va, cfg_local, width, lr, wd)
        Xte_n = channel_scale_apply(Xte, sx)
        pred = channel_scale_undo(predict_np(model, Xte_n, cfg_local.batch_size), sy)
        m = metrics_2ch(pred, Yte)
        return 0.5 * (m["rel_l2_re"] + m["rel_l2_im"]), meta

    rel_up, meta_up = one("up")
    rel_dn, meta_dn = one("down")

    return {
        "n_train": n_train,
        "rel_l2_mean": 0.5 * (rel_up + rel_dn),
        "rel_l2_up": rel_up,
        "rel_l2_down": rel_dn,
        "up_ckpt_loaded": meta_up["loaded"],
        "down_ckpt_loaded": meta_dn["loaded"],
    }


sizes = [64, 128, 192, 256]
size_rows = []

for n_train in tqdm(sizes, desc="Dataset-size sweep"):
    size_rows.append(run_with_train_size(
        n_train=n_train,
        cfg_base=cfg,
        width=int(best_hp["width"]),
        lr=float(best_hp["lr"]),
        wd=float(best_hp["weight_decay"]),
    ))


size_df = pd.DataFrame(size_rows)
display(size_df)

plt.figure(figsize=(6, 4))
plt.plot(size_df["n_train"], size_df["rel_l2_mean"], marker="o")
plt.xlabel("n_train")
plt.ylabel("mean rel_l2 (up/down)")
plt.title("Dataset-size sweep (single-source, cached checkpoints)")
plt.show()


## Notes

- First full run trains and writes artifacts to `artifacts/gemi_center_single/`.
- Later runs (including after kernel restart) load `best_hp` and any existing model checkpoints automatically.
- Delete artifacts manually only if you want to force a full retrain.
